# core

> What every other module needs and nothing else does: the error type, the one rule for reporting a caught exception, and the environment lookup.

These three lived in `leela/agent/__init__.py`. nbdev generates a package `__init__` rather
than reading one, so a name that every module imports needs a module of its own.

`env` exists because the harness has left leela. The four variables it reads were named
`LEELA_*`, and they are documented that way in leela's README, so the new name is preferred
and the old one still answers. Nobody has to be told to change anything.


In [ ]:
#| default_exp core

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import os

In [ ]:
#| export
class AgentError(Exception):
    "Something the harness refuses to do, as opposed to something that went wrong doing it."

In [ ]:
#| export
def agent_err(e):
    "How every harness surface reports a caught exception: `TypeName: message`."
    return f'{type(e).__name__}: {e}'

In [ ]:
#| export
#: Read first, so a machine that has both keeps the new name's answer.
ENV_PREFIX = 'RAMABANA_'
#: Still read, because leela documented these and breaking a documented variable to rename
#: a package is a cost paid by the user for a change that was not theirs.
ENV_FALLBACK = 'LEELA_'

def env(name, dflt=None):
    "`$RAMABANA_<name>`, else `$LEELA_<name>`, else `dflt`."
    return os.environ.get(ENV_PREFIX + name) or os.environ.get(ENV_FALLBACK + name) or dflt

## Tests


In [ ]:
print(agent_err(ValueError('nope')))
assert agent_err(ValueError('nope')) == 'ValueError: nope'
assert issubclass(AgentError, Exception)

In [ ]:
# The new name wins; the old one still answers; neither set means the default.
os.environ.pop('RAMABANA_MODEL', None); os.environ.pop('LEELA_MODEL', None)
print('neither    ->', env('MODEL', 'gemma-e2b'))
os.environ['LEELA_MODEL'] = 'from-leela'
print('leela only ->', env('MODEL'))
os.environ['RAMABANA_MODEL'] = 'from-ramabana'
print('both       ->', env('MODEL'))
assert env('MODEL') == 'from-ramabana'
del os.environ['RAMABANA_MODEL']
assert env('MODEL') == 'from-leela'
del os.environ['LEELA_MODEL']
assert env('MODEL', 'd') == 'd'